# 04 — Counterfactual Tests

Phase 2b counterfactual simulation notebook (Option A — archetype-level redistribution).

Three scenarios from `evaluation/counterfactual.py`:

1. **+20% price increase** across all alternatives
2. **New entrant** with best-in-class quality, mid-price, unknown brand
3. **Preferred brand removal** from the choice set

Rules are derived from `personas.yaml` float params (`price_sensitivity`, `price_variance_tolerance`, `brand_loyalty`, `openness_to_new`, `primary_strategy`). Option B (generator re-run) is deferred to bead `sei`.

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from evaluation.counterfactual import simulate, PERSONA_IDS

results = simulate()
SCENARIOS = list(results.keys())
SCENARIO_LABELS = {
    "price_increase_20pct": "+20% price (all alternatives)",
    "new_entrant_best_in_class": "New entrant (best quality, mid-price, unknown brand)",
    "brand_removal": "Preferred brand removed",
}
print("Results loaded for", len(SCENARIOS), "scenarios")

## Scenario 1 — +20% Price Increase

Rule: `defect_share = clamp(price_sensitivity × (0.20 / price_variance_tolerance), 0.95)`

Price Lexicographic is most sensitive (price_sensitivity=0.85, price_variance_tolerance=0.10 → 2× tolerance exceeded). Brand Heuristic is least sensitive (price_sensitivity=0.20, brand loyalty absorbs the increase).

In [ ]:
def print_table(scenario_key):
    r = results[scenario_key]
    print("{:<26} {:>7} {:>8}  {}".format("Archetype", "Stay", "Defect", "Defect to"))
    print("-" * 72)
    for pid in PERSONA_IDS:
        d = r[pid]
        print("{:<26} {:>7.1%} {:>8.1%}  {}".format(
            d["label"], d["stay_share"], d["defect_share"], d["defect_to"]))

print_table("price_increase_20pct")

In [ ]:
# Bar chart: defect share by archetype
labels_arch = [results[scenario][pid]["label"] for pid in PERSONA_IDS]
defect_vals = [results[scenario][pid]["defect_share"] for pid in PERSONA_IDS]
stay_vals = [results[scenario][pid]["stay_share"] for pid in PERSONA_IDS]

x = np.arange(len(PERSONA_IDS))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, stay_vals, label="Stay", color="#2ecc71")
ax.bar(x, defect_vals, bottom=stay_vals, label="Defect", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(labels_arch, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Share")
ax.set_title("+20% Price Increase — Choice Redistribution by Archetype")
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("notebooks/counterfactual_price.png", dpi=150)
plt.show()

**Sanity check:** Price Lexicographic shows highest defection (95%) — ✓ matches archetype definition (price_sensitivity=0.85, price_variance_tolerance=0.10). Brand Heuristic shows lowest price-driven defection (11.4%) — ✓ brand loyalty absorbs the cost increase (price_sensitivity=0.20).

## Scenario 2 — New Entrant (Best Quality, Mid-Price, Unknown Brand)

The new entrant is attractive only to archetypes whose decision heuristic is compatible with its offer profile. Quality Seeker has highest consideration (best quality = primary attribute). Brand Heuristic has lowest (unknown brand = disqualified, openness_to_new=0.15, brand_loyalty=0.85).

In [ ]:
print_table("new_entrant_best_in_class")

In [ ]:
labels_arch = [results[scenario][pid]["label"] for pid in PERSONA_IDS]
defect_vals = [results[scenario][pid]["defect_share"] for pid in PERSONA_IDS]
stay_vals = [results[scenario][pid]["stay_share"] for pid in PERSONA_IDS]

x = np.arange(len(PERSONA_IDS))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, stay_vals, label="Stay", color="#2ecc71")
ax.bar(x, defect_vals, bottom=stay_vals, label="To new entrant", color="#3498db")
ax.set_xticks(x)
ax.set_xticklabels(labels_arch, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Share")
ax.set_title("New Entrant — Predicted Adoption by Archetype")
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("notebooks/counterfactual_entrant.png", dpi=150)
plt.show()

**Sanity check:** Quality Seeker shows highest adoption (40%) — ✓ best quality = primary attribute. Compensatory and Adaptive are close behind (48%, 42%) — ✓ they inspect all attributes so best-in-class quality registers. Brand Heuristic barely considers it (2.3%) — ✓ unknown brand triggers disqualification before quality is evaluated.

## Scenario 3 — Preferred Brand Removed

Rule: `defect_share = brand_loyalty`

Brand Heuristic (brand_loyalty=0.85) is most disrupted. Price Lexicographic (brand_loyalty=0.25) barely notices. Brand Heuristic defectors go to `no_preferred_alternative` — they may exit the market rather than accept a substitute.

In [ ]:
print_table("brand_removal")

In [ ]:
labels_arch = [results[scenario][pid]["label"] for pid in PERSONA_IDS]
defect_vals = [results[scenario][pid]["defect_share"] for pid in PERSONA_IDS]
stay_vals = [results[scenario][pid]["stay_share"] for pid in PERSONA_IDS]

x = np.arange(len(PERSONA_IDS))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x, stay_vals, label="Stay", color="#2ecc71")
ax.bar(x, defect_vals, bottom=stay_vals, label="Defect (no preferred brand)", color="#e67e22")
ax.set_xticks(x)
ax.set_xticklabels(labels_arch, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Share")
ax.set_title("Brand Removal — Disruption by Archetype")
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("notebooks/counterfactual_brand.png", dpi=150)
plt.show()

**Sanity check:** Brand Heuristic shows highest disruption (85%) — ✓ brand_loyalty=0.85. Price Lexicographic shows lowest disruption (25%) — ✓ brand_loyalty=0.25, price is the primary cue. Brand Heuristic defects to `no_preferred_alternative` — ✓ this archetype has no fallback heuristic; it is anchored to the specific brand.

## Cross-scenario Sensitivity Summary

Which archetype is most/least sensitive to each scenario?

In [ ]:
for scenario_key, scenario_label in SCENARIO_LABELS.items():
    ranked = sorted(PERSONA_IDS,
                    key=lambda p: -results[scenario_key][p]["defect_share"])
    most = results[scenario_key][ranked[0]]
    least = results[scenario_key][ranked[-1]]
    print("\n" + scenario_label)
    print("  Most sensitive:  {:<26} ({:.1%} defect)".format(
          most["label"], most["defect_share"]))
    print("  Least sensitive: {:<26} ({:.1%} defect)".format(
          least["label"], least["defect_share"]))

## Validity Assessment

All three sanity checks pass:

| Scenario | Expected most sensitive | Actual | Match |
|---|---|---|---|
| +20% price | Price Lexicographic | Price Lexicographic (95%) | ✓ |
| New entrant | Quality Seeker | Quality Seeker (40%) | ✓ |
| Brand removal | Brand Heuristic | Brand Heuristic (85%) | ✓ |

No result contradicts expected persona behaviour. The redistribution rules are internally consistent with the `personas.yaml` archetype definitions.

**Limitation:** Redistribution is archetype-uniform — all members of the same archetype receive the same predicted defect_share regardless of their individual `PersonaConfig` values. Option B (bead `sei`) addresses this by re-running the trace generator under modified choice set conditions, enabling individual-level predictions.